# Observability for Agents

Companion notebook for the [Observability for Agents wiki page](https://ml-viz-ruby.vercel.app/wiki/agent-observability).

An agent's trace is the only record of what it *chose* to do — and the worst trajectories
complete without a single exception. We simulate a fleet of agent trajectories (deterministic,
offline), compute the **shape metrics** that catch behavioural failures (steps per task, tool
error rate, budget burn), see why cost grows **superlinearly** with trajectory length, and
finish by writing a loop detector.

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

rng = np.random.default_rng(7)

## 1 — Simulating agent trajectories

A trajectory is a list of steps. Each step is either an LLM call
(`('chat', tokens_in, tokens_out)`) or a tool call (`('tool', name, args, ok)`).
Most simulated agents behave; some get stuck repeating the same tool call with the same
arguments — the classic reasoning loop — and some hit a flaky tool.

In [ ]:
def make_trajectory(rng, looping=False, flaky_tool=False):
    steps, transcript = [], 400                      # tokens of fixed context
    n_actions = int(rng.integers(2, 6)) if not looping else 12
    for i in range(n_actions):
        out = int(rng.integers(80, 220))
        steps.append(('chat', transcript, out))      # each step re-sends the transcript
        transcript += out + 150                      # + tool result appended next
        if looping and i >= 3:
            steps.append(('tool', 'search_docs', 'q=refund policy', True))   # same call, forever
        else:
            name = ['search_docs', 'lookup_order', 'issue_refund'][int(rng.integers(0, 3))]
            ok = not (flaky_tool and name == 'lookup_order' and rng.random() < 0.7)
            steps.append(('tool', name, f'call_{i}', ok))
    steps.append(('chat', transcript, int(rng.integers(150, 300))))          # final answer
    return steps

fleet = ([make_trajectory(rng) for _ in range(60)]
       + [make_trajectory(rng, looping=True) for _ in range(4)]
       + [make_trajectory(rng, flaky_tool=True) for _ in range(16)])
print(f"{len(fleet)} trajectories; example lengths: {[len(t) for t in fleet[:5]]}")

## 2 — Shape metrics

Error-rate alerting sees nothing here: every trajectory "succeeds". The pathologies live in
the **shape** — steps per task, per-tool error rates, and token budget burn.

In [ ]:
def metrics(traj):
    chats = [s for s in traj if s[0] == 'chat']
    tools = [s for s in traj if s[0] == 'tool']
    return {
        'steps': len(traj),
        'tokens': sum(s[1] + s[2] for s in chats),
        'tool_errors': sum(1 for s in tools if not s[3]),
        'tool_calls': len(tools),
    }

M = [metrics(t) for t in fleet]
steps = np.array([m['steps'] for m in M])
tokens = np.array([m['tokens'] for m in M])
print(f"steps/task    p50={np.percentile(steps, 50):.0f}  p95={np.percentile(steps, 95):.0f}  max={steps.max()}")
print(f"tokens/task   p50={np.percentile(tokens, 50):,.0f}  p95={np.percentile(tokens, 95):,.0f}")

err_by_tool = {}
for t in fleet:
    for s in t:
        if s[0] == 'tool':
            ok_ct, tot = err_by_tool.get(s[1], (0, 0))
            err_by_tool[s[1]] = (ok_ct + (not s[3]), tot + 1)
for name, (errs, tot) in sorted(err_by_tool.items()):
    print(f"tool error rate  {name:<14} {errs / tot:5.1%}   ({errs}/{tot})")

One tool (`lookup_order`) is clearly broken for a subset of traffic, and the p95/max of
steps-per-task is far above the median — both invisible to average-based dashboards.

## 3 — Why long trajectories are expensive: superlinear tokens

Each step re-sends the growing transcript, so with fixed context $c_0$ and $\bar m$ tokens
added per step, total tokens are

$$\text{tokens}(S) \approx S c_0 + \frac{S(S+1)}{2}\,\bar m$$

— quadratic in trajectory length. A 2× longer trajectory ≈ 4× the tokens.

In [ ]:
S = np.arange(1, 41)
c0, m_bar = 400, 250
model = S * c0 + S * (S + 1) / 2 * m_bar

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(S, model / 1e3, color='#6366f1', label='tokens(S) — quadratic model')
chat_steps = np.array([sum(1 for s in t if s[0] == 'chat') for t in fleet])
ax.scatter(chat_steps, tokens / 1e3, s=18, color='#2dd4bf', alpha=0.7, label='simulated trajectories')
ax.set_xlabel('LLM steps in trajectory (S)'); ax.set_ylabel('total tokens (thousands)')
ax.set_title('Token cost grows superlinearly with trajectory length')
ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

The looping trajectories sit far up the curve — they cost ~10× the median task *before*
anyone notices anything is wrong. This is why **steps per task** and **budget burn** deserve
alerts, not just dashboards.

## ✏️ Your turn

**Concept recap.** A reasoning loop shows up in the trace as the same tool called with the
same arguments over and over. The **loop score** counts redundant tool calls: for each
`(tool, args)` pair that appears $n$ times, it contributes $n - 1$.

**Exercise.** Implement `loop_score(traj)` returning the total number of *redundant* tool
calls in one trajectory (0 for a healthy one). Then the alert is simply `loop_score(t) > 0`.

In [ ]:
def loop_score(traj):
    # TODO(you): count tool steps by their (name, args) pair;
    #            return sum over pairs of (count - 1)
    ...

# once implemented, inspect the fleet:
# sorted(loop_score(t) for t in fleet)[-6:]

In [ ]:
# This assert cell passes silently when your implementation is correct.
healthy = [('chat', 400, 100), ('tool', 'a', 'x', True), ('tool', 'a', 'y', True)]
loopy = healthy + [('tool', 'a', 'x', True)] * 3
assert loop_score(healthy) == 0
assert loop_score(loopy) == 3          # ('a','x') appears 4 times -> 3 redundant
flagged = [i for i, t in enumerate(fleet) if loop_score(t) > 0]
assert len(flagged) == 4, f"expected the 4 planted loopers, flagged {len(flagged)}"
print('✓ loop_score flags exactly the planted looping trajectories')

<details>
<summary>Solution</summary>

```python
def loop_score(traj):
    counts = {}
    for s in traj:
        if s[0] == 'tool':
            key = (s[1], s[2])
            counts[key] = counts.get(key, 0) + 1
    return sum(n - 1 for n in counts.values())
```

In production you'd compute this per-trace at ingestion and alert on `loop_score > 0`
together with `steps > p95` and `budget_burn > 0.8 × ceiling` — shape alerts for a system
that fails without ever throwing.
</details>